#Projekt hurtowni danych

##Wstępne założenia, testy oraz opis problemów związanych ze specyfiką danych flights

###Założenia:
- **Istnieje możliwość bezpośredniego edytowania danych źródłowych** <br>
Przykład: Istnieje jeden plik, w którym dane są regularnie dopisywane. W takim wypadku przydatna będzie kolumna timestamp/loaded_flag, aby w przypadku przyrostowego ładowania nie było potrzeby sprawdzania w kółko całego pliku, a jedynie nowych rekordów.
W załączonym zbiorze takie timestampy nie są śledzone, co więcej nie mamy unikalnych idnetyfikatorów dla każdego z rekordów, załóżmy więc, że takie informacje zostają dopisane przy załadowaniu danych.

Z powyższego założenia wynika więc kolejne: <br>
- **Wczytywany jest cały czas ten sam plik, do którego dodawne są nowe rekordy lub istniejące są edytowane. Przy edycji rekordu flaga wczytywania ustawiana jest na 0, podobnie przy wpisywaniu nowego rekordu. Po wczytaniu danych do bazy, wartość dla flagi zmienia się na 1** <br>


Uwaga: Jeśli powyższe założenie jest "nielegalne" dalej możliwym rozwiązaniem będzie dodawanie jedynie nowych rekordów, jednak będzie wymagać to bardzo dużej liczby porównań, a więc takie rozwiązanie na pewno nie będzie bliskie optymalnemu. Poza tym, przez brak uniwerslanych identyfikatorów, znalezienie edytowanych wierszy może być niejednoznaczne.

Już wcześniej wspomnianym poważnym problemem jest **brak jednoznaczengo identyfikatora rekordu w danych źródłowych dla flights** (nie ma kolumny id). <br>
Przez to metoda wyszukiwania duplikatów oraz edycji rekordów może wymagać porównywania ze sobą wartości we wszystkich kolumnach. Na tym etapie musimy przyjąć kolejne założenie, które pozwala wyznaczyć id poszczególnego rekordu.

Poniżej samo sprawdzenie czy i dla jakich kolumn takie podejście będzie poprawne

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
flights_schema = StructType([
    StructField("YEAR", IntegerType(), True),
    StructField("MONTH", IntegerType(), True),
    StructField("DAY", IntegerType(), True),
    StructField("DAY_OF_WEEK", IntegerType(), True),
    StructField("AIRLINE", StringType(), True),
    StructField("FLIGHT_NUMBER", StringType(), True),
    StructField("TAIL_NUMBER", StringType(), True),
    StructField("ORIGIN_AIRPORT", StringType(), True),
    StructField("DESTINATION_AIRPORT", StringType(), True),
    StructField("SCHEDULED_DEPARTURE", IntegerType(), True),
    StructField("DEPARTURE_TIME", IntegerType(), True),
    StructField("DEPARTURE_DELAY", IntegerType(), True),
    StructField("TAXI_OUT", IntegerType(), True),
    StructField("WHEELS_OFF", IntegerType(), True),
    StructField("SCHEDULED_TIME", IntegerType(), True),
    StructField("ELAPSED_TIME", IntegerType(), True),
    StructField("AIR_TIME", IntegerType(), True),
    StructField("DISTANCE", IntegerType(), True),
    StructField("WHEELS_ON", IntegerType(), True),
    StructField("TAXI_IN", IntegerType(), True),
    StructField("SCHEDULED_ARRIVAL", IntegerType(), True),
    StructField("ARRIVAL_TIME", IntegerType(), True),
    StructField("ARRIVAL_DELAY", IntegerType(), True),
    StructField("DIVERTED", IntegerType(), True),
    StructField("CANCELLED", IntegerType(), True),
    StructField("CANCELLATION_REASON", StringType(), True),
    StructField("AIR_SYSTEM_DELAY", IntegerType(), True),
    StructField("SECURITY_DELAY", IntegerType(), True),
    StructField("AIRLINE_DELAY", IntegerType(), True),
    StructField("LATE_AIRCRAFT_DELAY", IntegerType(), True),
    StructField("WEATHER_DELAY", IntegerType(), True) 
])

In [0]:
flights_df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(flights_schema) \
    .load("/FileStore/tables/flights.csv")

In [0]:
flights_df.toPandas().head(4)

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,None,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,None,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,None,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,None,NaN,NaN,NaN,NaN,NaN


Liczba rekordów

In [0]:
total_rows = flights_df.count()

Założenie: unikalny rekord to taki, który ma różne jednocześnie: rok, miesiąc, dzień, godzine planowanego wylotu, numer lotu, "numer" samolotu

Liczba unikalnych wierszy

In [0]:
unique_rows = flights_df.select("YEAR", "MONTH", "DAY", "FLIGHT_NUMBER","TAIL_NUMBER", "SCHEDULED_DEPARTURE").distinct().count()

In [0]:
print(f"Ogólna liczba wierszy: {total_rows}")
print(f"Liczba unikalnych zestawów: {unique_rows}")

Ogólna liczba wierszy: 5819079
Liczba unikalnych zestawów: 5819077


Istnieją rekordy, które nawet dla tak ogranicznego podziału są różnymi "faktami".

In [0]:
duplicates_df = flights_df.groupBy(
    "YEAR", "MONTH", "DAY", "FLIGHT_NUMBER","TAIL_NUMBER", "SCHEDULED_DEPARTURE"
).count().filter(F.col("count") > 1)

full_duplicates_df = flights_df.join(
    duplicates_df,
    on=["YEAR", "MONTH", "DAY", "FLIGHT_NUMBER", "TAIL_NUMBER", "SCHEDULED_DEPARTURE"],
    how="inner"
)

full_duplicates_df.toPandas().head(10)

,YEAR,MONTH,DAY,FLIGHT_NUMBER,TAIL_NUMBER,SCHEDULED_DEPARTURE,DAY_OF_WEEK,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,...,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,count
0,2015,10,16,5660,N14204,704,5,EV,11049,12266,...,123.0,0,0,None,0.0,0.0,123.0,0.0,0.0,2
1,2015,10,16,5660,N14204,704,5,EV,14814,12266,...,NaN,1,0,None,NaN,NaN,NaN,NaN,NaN,2


Pierwsze dwa rekordy, dla których zestaw wybranych wartości jest taki sam, dotyczy przekierowanego lotu. Jest to pojedynczy przypadek, warto jednak zwrócić uwagę że jest to ten sam lot, ale jak wynika z danych, został przekierowany na inne lotnisko. Taka niejednoznaczność może potencjanie zaburzać statystyki podsumowujące. W tym rozwiązaniu loty przekierowane będą traktowane jako różne rekordy, jednak z uwzględnieniem, że jest to technicznie ten sam lot już przy statystykach podsumowujących. Dodajemy do kryteriów tworzenia unikalnego rekrodu również kolumnę "DIVERTED".

In [0]:
unique_rows = flights_df.select("YEAR", "MONTH", "DAY", "FLIGHT_NUMBER","TAIL_NUMBER", "SCHEDULED_DEPARTURE", "DIVERTED").distinct().count()

In [0]:
print(f"Ogólna liczba wierszy: {total_rows}")
print(f"Liczba unikalnych zestawów: {unique_rows}")

Ogólna liczba wierszy: 5819079
Liczba unikalnych zestawów: 5819078


Dalej pozostaje para rekordów, która różni się wartościami dla innych kolumn.

In [0]:
#Zamieniamy wartości NaN na 0/Unknown, ponieważ w innym przypadku, przy porównaniu, są traktowane jako inne rekordy.
flights_df = flights_df.fillna({
    col: 0 if flights_df.schema[col].dataType.typeName() in ["integer", "double"] else "UNKNOWN"
    for col in flights_df.columns
})


In [0]:
duplicates_df = flights_df.groupBy(
    "YEAR", "MONTH", "DAY", "FLIGHT_NUMBER", "TAIL_NUMBER", "SCHEDULED_DEPARTURE", "DIVERTED"
).count().filter(F.col("count") > 1)

full_duplicates_df = flights_df.join(
    duplicates_df,
    on=["YEAR", "MONTH", "DAY", "FLIGHT_NUMBER", "TAIL_NUMBER", "SCHEDULED_DEPARTURE", "DIVERTED"],
    how="inner"
)

columns_to_check = [col for col in flights_df.columns if col not in ["YEAR", "MONTH", "DAY", "FLIGHT_NUMBER", "TAIL_NUMBER", "SCHEDULED_DEPARTURE", "DIVERTED"]]

result_df = full_duplicates_df.sort(
    "YEAR", "MONTH", "DAY", "FLIGHT_NUMBER", "SCHEDULED_DEPARTURE", "DIVERTED"
)

result_df.show(truncate=False)


+----+-----+---+-------------+-----------+-------------------+--------+-----------+-------+--------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+-----+
|YEAR|MONTH|DAY|FLIGHT_NUMBER|TAIL_NUMBER|SCHEDULED_DEPARTURE|DIVERTED|DAY_OF_WEEK|AIRLINE|ORIGIN_AIRPORT|DESTINATION_AIRPORT|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|count|
+----+-----+---+-------------+-----------+-------------------+--------+-----------+-------+--------------+-------------------+--------------+---------------+--------+----------+--------------+--

In [0]:
columns_to_display = result_df.columns[10:20]

result_df.select(columns_to_display).toPandas().head(2)


,DESTINATION_AIRPORT,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON
0,CLT,0,0,0,0,232,0,0,1520,0
1,SJU,0,0,0,0,25,0,0,68,0


Powyższe rekordy wyglądają jakby opisywany był ten sam "fakt", szczególnie ze względu na to że posiadją ten sam fligh_number, date i godzine. Okazuje się jednak, że takich rekordów(które mają ten sam numer lotu i czas) jest dosyć dużo, a więc niekoniecznie są błędne, a raczej wygląda na to że sam sposób wpisywania tych danych jest mylący. Widać więc, że mogą istnieć sytacje, gdzie numer lotu, dokładny czas oraz trasa mogą się pokrywać, a jednocześnie opisywać zupłenie inny fakt. Stąd do tworzenia uniwerslanego identyfikatora kluczowe będzie użycie także kodu konkretnego samolotu (który ewidentnie nie może być w tym samym czasie w dwóćh różnych miejscach), pomimo że w kolumnie tail_number powiaja się kilka rekordów z brakującą wartością.

In [0]:
result_df.toPandas().head(2)


,YEAR,MONTH,DAY,FLIGHT_NUMBER,TAIL_NUMBER,SCHEDULED_DEPARTURE,DIVERTED,DAY_OF_WEEK,AIRLINE,ORIGIN_AIRPORT,...,ARRIVAL_TIME,ARRIVAL_DELAY,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,count
0,2015,8,29,803,UNKNOWN,1435,0,6,AA,STT,...,0,0,1,A,0,0,0,0,0,2
1,2015,8,29,803,UNKNOWN,1435,0,6,AA,STT,...,0,0,1,A,0,0,0,0,0,2


**Na podstawie powyższych testów można założyć więc, że kolumny "YEAR", "MONTH", "DAY", "FLIGHT_NUMBER", "TAIL_NUMBER", "SCHEDULED_DEPARTURE", "DESTINATION_AIRPORT", "DIVERTED" mogą być wystarczające do wyznaczenia uniwersalnych id.**

Sprawdzenie czy origin i destination airport zawsze mają taki sam dystans, a więc czy będzie możliwość stworzenia z nich jednej z tabeli wymiarów.

In [0]:
distance_check_df = flights_df.groupBy("ORIGIN_AIRPORT", "DESTINATION_AIRPORT") \
    .agg(F.countDistinct("DISTANCE").alias("unique_distance_count"))

inconsistent_distances = distance_check_df.filter(F.col("unique_distance_count") > 1)
inconsistent_distances.show()

count_inconsistent = inconsistent_distances.count() 
print(f'Liczba niespójnych par origin-destination: {count_inconsistent}')


+--------------+-------------------+---------------------+
|ORIGIN_AIRPORT|DESTINATION_AIRPORT|unique_distance_count|
+--------------+-------------------+---------------------+
|           ORD|                BUF|                    2|
|           CVG|                ORD|                    2|
|           ORD|                CAE|                    2|
|           FLL|                SFO|                    2|
|           BQN|                FLL|                    2|
|           DFW|                MEM|                    2|
|           AUS|                ORD|                    2|
|           TYS|                DFW|                    2|
|           CWA|                ORD|                    2|
|           FLL|                BUF|                    2|
|           BUF|                ORD|                    2|
|           FLL|                MDW|                    2|
|           ORD|                GSP|                    2|
|           BUF|                FLL|                    

In [0]:
inconsistent_distances_full = inconsistent_distances.join(flights_df, ["ORIGIN_AIRPORT", "DESTINATION_AIRPORT"], "inner")
distinct_distances = inconsistent_distances_full.groupBy("ORIGIN_AIRPORT", "DESTINATION_AIRPORT") \
    .agg(F.collect_set("DISTANCE").alias("DISTANCES")) 
distinct_distances.show(truncate=False)

+--------------+-------------------+------------+
|ORIGIN_AIRPORT|DESTINATION_AIRPORT|DISTANCES   |
+--------------+-------------------+------------+
|CVG           |ORD                |[264, 265]  |
|ORD           |BUF                |[473, 474]  |
|ORD           |CAE                |[666, 667]  |
|FLL           |SFO                |[2583, 2584]|
|BQN           |FLL                |[982, 983]  |
|DFW           |MEM                |[431, 432]  |
|AUS           |ORD                |[978, 977]  |
|TYS           |DFW                |[771, 772]  |
|CWA           |ORD                |[212, 213]  |
|BUF           |ORD                |[473, 474]  |
|FLL           |BUF                |[1166, 1165]|
|FLL           |MDW                |[1166, 1167]|
|ORD           |GSP                |[578, 577]  |
|BUF           |FLL                |[1166, 1165]|
|PHX           |DRO                |[353, 351]  |
|SDF           |ORD                |[287, 286]  |
|ORD           |CWA                |[212, 213]  |


Różnice można uznać za pomijalne, a więc pod tym kątem stworzenie tabeli wymiarów z wykorzystaniem tych kolumn wydaje się być poprawne. Poniżej sprawdzenie ile jest wystąpień takich par ORIGIN-DESTINATION i czy jest sens zapisywanie takich połączeń jako osobny wymiar.

In [0]:
pair_counts = flights_df.groupBy("ORIGIN_AIRPORT", "DESTINATION_AIRPORT").agg(F.count("*").alias("count"))
pair_counts.show(truncate=False)


+--------------+-------------------+-----+
|ORIGIN_AIRPORT|DESTINATION_AIRPORT|count|
+--------------+-------------------+-----+
|BQN           |MCO                |441  |
|PHL           |MCO                |4869 |
|MCI           |IAH                |1698 |
|SPI           |ORD                |998  |
|SNA           |PHX                |3846 |
|LBB           |DEN                |618  |
|ORD           |PDX                |2149 |
|EWR           |STT                |239  |
|ATL           |GSP                |2470 |
|MCI           |MKE                |612  |
|PBI           |DCA                |978  |
|SMF           |BUR                |2092 |
|MDW           |MEM                |628  |
|LAS           |LIT                |334  |
|TPA           |ACY                |335  |
|DSM           |EWR                |191  |
|FSD           |ATL                |329  |
|SJC           |LIH                |190  |
|CLE           |SJU                |43   |
|CPR           |DEN                |956  |
+----------

Dla każdego połączenia pojawia się względnie duża liczba rekordów, wynika z tego że rzeczywiście dobrym wyborem będzie przedstawineie tych danych w tabeli wymiarów.

##Właściwa Implementacja

In [0]:
dbutils.fs.ls('/FileStore/tables/flights-2.csv')
dbutils.fs.ls('/FileStore/tables/airports-1.csv')
dbutils.fs.ls('/FileStore/tables/airlines-1.csv')

Out[381]: [FileInfo(path='dbfs:/FileStore/tables/airlines-1.csv', name='airlines-1.csv', size=414, modificationTime=1733985206000)]

In [0]:
%sql
CREATE DATABASE data_warehouse;

In [0]:
%sql
USE data_warehouse;

##Wczytywanie danych do wartstwy staging

Dane będą przygtowane, przekształcone oraz wstępnie przeanalizowane za pomocą PySpark dataframes, który zapewnia szybkie obliczenia oraz dużo przydtanych funkcji.

###Flights

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType
from pyspark.sql.functions import col, when
import pandas as pd
from pyspark.sql import functions as F

In [0]:
flights_schema = StructType([
    StructField("YEAR", IntegerType(), False),
    StructField("MONTH", IntegerType(), False),
    StructField("DAY", IntegerType(), False),
    StructField("DAY_OF_WEEK", IntegerType(), True),
    StructField("AIRLINE", StringType(), True),
    StructField("FLIGHT_NUMBER", StringType(), False),
    StructField("TAIL_NUMBER", StringType(), False),
    StructField("ORIGIN_AIRPORT", StringType(), False),
    StructField("DESTINATION_AIRPORT", StringType(), False),
    StructField("SCHEDULED_DEPARTURE", IntegerType(), False),
    StructField("DEPARTURE_TIME", IntegerType(), True),
    StructField("DEPARTURE_DELAY", IntegerType(), True),
    StructField("TAXI_OUT", IntegerType(), True),
    StructField("WHEELS_OFF", IntegerType(), True),
    StructField("SCHEDULED_TIME", IntegerType(), True),
    StructField("ELAPSED_TIME", IntegerType(), True),
    StructField("AIR_TIME", IntegerType(), True),
    StructField("DISTANCE", IntegerType(), True),
    StructField("WHEELS_ON", IntegerType(), True),
    StructField("TAXI_IN", IntegerType(), True),
    StructField("SCHEDULED_ARRIVAL", IntegerType(), True),
    StructField("ARRIVAL_TIME", IntegerType(), True),
    StructField("ARRIVAL_DELAY", IntegerType(), True),
    StructField("DIVERTED", IntegerType(), False),
    StructField("CANCELLED", IntegerType(), True),
    StructField("CANCELLATION_REASON", StringType(), True),
    StructField("AIR_SYSTEM_DELAY", IntegerType(), True),
    StructField("SECURITY_DELAY", IntegerType(), True),
    StructField("AIRLINE_DELAY", IntegerType(), True),
    StructField("LATE_AIRCRAFT_DELAY", IntegerType(), True),
    StructField("WEATHER_DELAY", IntegerType(), True),
    StructField("LOADED_FLAG", IntegerType(), False)
])

flights_df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(flights_schema) \
    .load("/FileStore/tables/flights-2.csv")

In [0]:
flights_df.toPandas().head(4)

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,LOADED_FLAG
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,-22.0,0,0,None,NaN,NaN,NaN,NaN,NaN,0
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,-9.0,0,0,None,NaN,NaN,NaN,NaN,NaN,0
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,5.0,0,0,None,NaN,NaN,NaN,NaN,NaN,0
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,-9.0,0,0,None,NaN,NaN,NaN,NaN,NaN,0


Sprawdzenie braków danych. Nie istnieje żadna kolumna z samymi wartościami NaN. W końcowej bazie więc pozostawiam wszystkie. 

In [0]:
nan_percentages = flights_df.select(
    [
        (F.count(F.when(F.col(c).isNull(), c)) / F.count("*") * 100).alias(c)
        for c in flights_df.columns
    ]
)
nan_percentages_df = nan_percentages.toPandas().T.reset_index()
nan_percentages_df.columns = ["Column", "Percentage of NaN Values"]

print(nan_percentages_df.to_string(index=False))


             Column  Percentage of NaN Values
               YEAR                  0.000000
              MONTH                  0.000000
                DAY                  0.000000
        DAY_OF_WEEK                  0.000000
            AIRLINE                  0.000000
      FLIGHT_NUMBER                  0.000000
        TAIL_NUMBER                  0.739098
     ORIGIN_AIRPORT                  0.000000
DESTINATION_AIRPORT                  0.000000
SCHEDULED_DEPARTURE                  0.000000
     DEPARTURE_TIME                  0.000000
    DEPARTURE_DELAY                  3.768448
           TAXI_OUT                  3.836540
         WHEELS_OFF                  0.000000
     SCHEDULED_TIME                  0.000191
       ELAPSED_TIME                  4.107575
           AIR_TIME                  4.107575
           DISTANCE                  0.000000
          WHEELS_ON                  0.000000
            TAXI_IN                  3.938297
  SCHEDULED_ARRIVAL               

Nieczytelnie zapisane godziny.

In [0]:
flights_df.toPandas().iloc[:, 9:20].head(4)

,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN
0,5,2354.0,-11.0,21.0,15.0,205.0,194.0,169.0,1448,404.0,4.0
1,10,2.0,-8.0,12.0,14.0,280.0,279.0,263.0,2330,737.0,4.0
2,20,18.0,-2.0,16.0,34.0,286.0,293.0,266.0,2296,800.0,11.0
3,20,15.0,-5.0,15.0,30.0,285.0,281.0,258.0,2342,748.0,8.0


Sformatowanie godzin. W pyspark nie istnieje typ 'time', godziny więc pozostają w formacie String, ale według schematu zapisywania godzin, co poprawia czytelność tych wartości. 

In [0]:
def convert_time_col(column):
    return F.when(F.length(F.col(column)) == 1, F.format_string("00:0%s", F.col(column))) \
            .when(F.length(F.col(column)) == 2, F.format_string("00:%s", F.col(column))) \
            .when(F.length(F.col(column)) == 3, F.format_string("0%s:%s", F.col(column).substr(1,1), F.col(column).substr(2,2))) \
            .otherwise(F.format_string("%s:%s", F.col(column).substr(1,2), F.col(column).substr(3,2)))


In [0]:
for column in ["SCHEDULED_DEPARTURE", "DEPARTURE_TIME", "WHEELS_OFF", "WHEELS_ON", "SCHEDULED_ARRIVAL", "ARRIVAL_TIME"]:
    flights_df = flights_df.withColumn(column, convert_time_col(column))

In [0]:
flights_df.toPandas().iloc[:, 9:20].head(4)

,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN
0,00:05,23:54,-11.0,21.0,00:15,205.0,194.0,169.0,1448,04:04,4.0
1,00:10,00:02,-8.0,12.0,00:14,280.0,279.0,263.0,2330,07:37,4.0
2,00:20,00:18,-2.0,16.0,00:34,286.0,293.0,266.0,2296,08:00,11.0
3,00:20,00:15,-5.0,15.0,00:30,285.0,281.0,258.0,2342,07:48,8.0


In [0]:
%sql
DROP TABLE FLIGHTS_STAGING;
CREATE TABLE FLIGHTS_STAGING (
  YEAR INT NOT NULL,
  MONTH INT NOT NULL,
  DAY INT NOT NULL,
  DAY_OF_WEEK INT,
  AIRLINE STRING,
  FLIGHT_NUMBER STRING NOT NULL,
  TAIL_NUMBER STRING,
  ORIGIN_AIRPORT STRING NOT NULL,
  DESTINATION_AIRPORT STRING NOT NULL,
  SCHEDULED_DEPARTURE STRING NOT NULL, 
  DEPARTURE_TIME STRING,     
  DEPARTURE_DELAY INT,
  TAXI_OUT INT,
  WHEELS_OFF STRING,      
  SCHEDULED_TIME INT,
  ELAPSED_TIME INT,
  AIR_TIME INT,
  DISTANCE INT,
  WHEELS_ON STRING,        
  TAXI_IN INT,
  SCHEDULED_ARRIVAL STRING,  
  ARRIVAL_TIME STRING,       
  ARRIVAL_DELAY INT,
  DIVERTED BOOLEAN NOT NULL,
  CANCELLED BOOLEAN,
  CANCELLATION_REASON STRING,
  AIR_SYSTEM_DELAY INT,
  SECURITY_DELAY INT,
  AIRLINE_DELAY INT,
  LATE_AIRCRAFT_DELAY INT,
  WEATHER_DELAY INT,
  LOADED_FLAG BOOLEAN NOT NULL
);


In [0]:
for column, dtype in flights_df.dtypes:
    print(f"Kolumna: {column}, Typ danych: {dtype}")

Kolumna: YEAR, Typ danych: int
Kolumna: MONTH, Typ danych: int
Kolumna: DAY, Typ danych: int
Kolumna: DAY_OF_WEEK, Typ danych: int
Kolumna: AIRLINE, Typ danych: string
Kolumna: FLIGHT_NUMBER, Typ danych: string
Kolumna: TAIL_NUMBER, Typ danych: string
Kolumna: ORIGIN_AIRPORT, Typ danych: string
Kolumna: DESTINATION_AIRPORT, Typ danych: string
Kolumna: SCHEDULED_DEPARTURE, Typ danych: string
Kolumna: DEPARTURE_TIME, Typ danych: string
Kolumna: DEPARTURE_DELAY, Typ danych: int
Kolumna: TAXI_OUT, Typ danych: int
Kolumna: WHEELS_OFF, Typ danych: string
Kolumna: SCHEDULED_TIME, Typ danych: int
Kolumna: ELAPSED_TIME, Typ danych: int
Kolumna: AIR_TIME, Typ danych: int
Kolumna: DISTANCE, Typ danych: int
Kolumna: WHEELS_ON, Typ danych: string
Kolumna: TAXI_IN, Typ danych: int
Kolumna: SCHEDULED_ARRIVAL, Typ danych: string
Kolumna: ARRIVAL_TIME, Typ danych: string
Kolumna: ARRIVAL_DELAY, Typ danych: int
Kolumna: DIVERTED, Typ danych: int
Kolumna: CANCELLED, Typ danych: int
Kolumna: CANCELLATION_

Konwersja danych integer na wartości boolean - True/False

In [0]:
from pyspark.sql.functions import col
bool_cols=["DIVERTED", "CANCELLED", "LOADED_FLAG"]
for bool_col in bool_cols:
    flights_df = flights_df.withColumn(bool_col, when(col(bool_col) == 1, True).otherwise(False))

In [0]:
flights_df.write.mode("append").saveAsTable("FLIGHTS_STAGING")

In [0]:
%sql
SELECT * FROM FLIGHTS_STAGING LIMIT 10;

YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,LOADED_FLAG
2015,1,1,4,AS,98,N407AS,ANC,SEA,00:05,23:54,-11,21,00:15,205,194,169,1448,04:04,4,04:30,04:08,-22,false,false,null,null,null,null,null,null,false
2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,00:10,00:02,-8,12,00:14,280,279,263,2330,07:37,4,07:50,07:41,-9,false,false,null,null,null,null,null,null,false
2015,1,1,4,US,840,N171US,SFO,CLT,00:20,00:18,-2,16,00:34,286,293,266,2296,08:00,11,08:06,08:11,5,false,false,null,null,null,null,null,null,false
2015,1,1,4,AA,258,N3HYAA,LAX,MIA,00:20,00:15,-5,15,00:30,285,281,258,2342,07:48,8,08:05,07:56,-9,false,false,null,null,null,null,null,null,false
2015,1,1,4,AS,135,N527AS,SEA,ANC,00:25,00:24,-1,11,00:35,235,215,199,1448,02:54,5,03:20,02:59,-21,false,false,null,null,null,null,null,null,false
2015,1,1,4,DL,806,N3730B,SFO,MSP,00:25,00:20,-5,18,00:38,217,230,206,1589,06:04,6,06:02,06:10,8,false,false,null,null,null,null,null,null,false
2015,1,1,4,NK,612,N635NK,LAS,MSP,00:25,00:19,-6,11,00:30,181,170,154,1299,05:04,5,05:26,05:09,-17,false,false,null,null,null,null,null,null,false
2015,1,1,4,US,2013,N584UW,LAX,CLT,00:30,00:44,14,13,00:57,273,249,228,2125,07:45,8,08:03,07:53,-10,false,false,null,null,null,null,null,null,false
2015,1,1,4,AA,1112,N3LAAA,SFO,DFW,00:30,00:19,-11,17,00:36,195,193,173,1464,05:29,3,05:45,05:32,-13,false,false,null,null,null,null,null,null,false
2015,1,1,4,DL,1173,N826DN,LAS,ATL,00:30,00:33,3,12,00:45,221,203,186,1747,06:51,5,07:11,06:56,-15,false,false,null,null,null,null,null,null,false


###Airports

In [0]:
airports_schema = StructType([
    StructField("IATA_CODE", StringType(), False),
    StructField("AIRPORT", StringType(), False),
    StructField("CITY", StringType(), True),
    StructField("STATE", StringType(), True),
    StructField("COUNTRY", StringType(), True),
    StructField("LATITUDE", FloatType(), True),
    StructField("LONGITUDE", FloatType(), True),
    StructField("LOADED_FLAG", IntegerType(), False)])

airports_df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(airports_schema) \
    .load("/FileStore/tables/airports-1.csv")

In [0]:
airlines_schema = StructType([
    StructField("IATA_CODE", StringType(), False),
    StructField("AIRLINE", StringType(), False),
    StructField("LOADED_FLAG", IntegerType(), False)])

airlines_df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(airlines_schema) \
    .load("/FileStore/tables/airlines-1.csv")

In [0]:
%sql
DROP TABLE AIRPORTS_STAGING;
CREATE TABLE AIRPORTS_STAGING (
  IATA_CODE STRING NOT NULL,
  AIRPORT STRING NOT NULL,
  CITY STRING,
  STATE STRING,
  COUNTRY STRING,
  LATITUDE FLOAT,
  LONGITUDE FLOAT,
  LOADED_FLAG BOOLEAN NOT NULL
);


In [0]:
from pyspark.sql.functions import col, when
airports_df = airports_df.withColumn("LOADED_FLAG", when(col("LOADED_FLAG") == 1, True).otherwise(False))

In [0]:
airports_df.write.mode("append").saveAsTable("AIRPORTS_STAGING")

In [0]:
%sql
SELECT * FROM AIRPORTS_STAGING LIMIT 10;

IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE,LOADED_FLAG
ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.4404,false
ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.6819,false
ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919,false
ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183,false
ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447,false
ACK,Nantucket Memorial Airport,Nantucket,MA,USA,41.25305,-70.06018,false
ACT,Waco Regional Airport,Waco,TX,USA,31.61129,-97.23052,false
ACV,Arcata Airport,Arcata/Eureka,CA,USA,40.97812,-124.10862,false
ACY,Atlantic City International Airport,Atlantic City,NJ,USA,39.45758,-74.57717,false
ADK,Adak Airport,Adak,AK,USA,51.87796,-176.64603,false


###Airlines

In [0]:
airlines_df=spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/FileStore/tables/airlines-1.csv")
airlines_df.printSchema()


root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- LOADED_FLAG: integer (nullable = true)



Konwersja danych integer na wartości boolean - True/False

In [0]:
from pyspark.sql.functions import col, when
airlines_df = airlines_df.withColumn("LOADED_FLAG", when(col("LOADED_FLAG") == 1, True).otherwise(False))


In [0]:
airlines_df.write.mode("append").saveAsTable("AIRLINES_STAGING")

In [0]:
%sql
SELECT * FROM AIRLINES_STAGING LIMIT 10;

IATA_CODE,AIRLINE,LOADED_FLAG
UA,United Air Lines Inc.,false
AA,American Airlines Inc.,false
US,US Airways Inc.,false
F9,Frontier Airlines Inc.,false
B6,JetBlue Airways,false
OO,Skywest Airlines Inc.,false
AS,Alaska Airlines Inc.,false
NK,Spirit Air Lines,false
WN,Southwest Airlines Co.,false
DL,Delta Air Lines Inc.,false


##Wczytywanie danych do warstwy curated

###Airlines

In [0]:
%sql
DROP TABLE AIRLINES_CURATED;
CREATE TABLE AIRLINES_CURATED (
    AIRLINE_ID INT NOT NULL,
    IATA_CODE STRING NOT NULL,
    AIRLINE STRING NOT NULL,
    START_DATE DATE NOT NULL,
    END_DATE DATE,
    IS_ACTIVE BOOLEAN NOT NULL
);


In [0]:
%sql
-- Krok 1: Oznaczam stare rekordy jako nieaktywne
MERGE INTO AIRLINES_CURATED AS c
USING (
    SELECT 
        ROW_NUMBER() OVER (PARTITION BY IATA_CODE ORDER BY IATA_CODE) AS AIRLINE_ID,
        IATA_CODE,
        AIRLINE,
        LOADED_FLAG
    FROM AIRLINES_STAGING
    WHERE LOADED_FLAG = FALSE
) AS s
ON c.IATA_CODE = s.IATA_CODE AND c.IS_ACTIVE = TRUE
WHEN MATCHED AND (
    c.AIRLINE <> s.AIRLINE
) THEN 
    UPDATE SET 
        END_DATE = '2024-12-10',
        IS_ACTIVE = FALSE;

-- Krok 2: Wstawiam nowe rekordy
INSERT INTO AIRLINES_CURATED (
    AIRLINE_ID, IATA_CODE, AIRLINE, START_DATE, END_DATE, IS_ACTIVE
)
WITH new_records AS (
    SELECT 
        ROW_NUMBER() OVER (PARTITION BY IATA_CODE ORDER BY IATA_CODE) + 
            (SELECT COALESCE(MAX(AIRLINE_ID), 0) FROM AIRLINES_CURATED) AS AIRLINE_ID,
        IATA_CODE,
        AIRLINE
    FROM AIRLINES_STAGING s
    WHERE LOADED_FLAG = FALSE
)
SELECT 
    n.AIRLINE_ID,
    n.IATA_CODE,
    n.AIRLINE,
    '2024-12-10' as START_DATE,
    NULL as END_DATE,
    TRUE as IS_ACTIVE
FROM new_records n
LEFT JOIN AIRLINES_CURATED c 
    ON n.IATA_CODE = c.IATA_CODE AND c.IS_ACTIVE = TRUE
WHERE 
    c.IATA_CODE IS NULL  -- dla nowych rekordów
    OR c.AIRLINE <> n.AIRLINE;  -- dla zmienionych rekordów

-- Aktualizacja flag w staging
UPDATE AIRLINES_STAGING
SET LOADED_FLAG = TRUE
WHERE LOADED_FLAG = FALSE;


num_affected_rows
28


In [0]:
%sql
SELECT * FROM AIRLINES_CURATED LIMIT 10;

AIRLINE_ID,IATA_CODE,AIRLINE,START_DATE,END_DATE,IS_ACTIVE
1,AA,American Airlines Inc.,2024-12-10,null,true
2,AA,American Airlines Inc.,2024-12-10,null,true
1,AS,Alaska Airlines Inc.,2024-12-10,null,true
2,AS,Alaska Airlines Inc.,2024-12-10,null,true
1,B6,JetBlue Airways,2024-12-10,null,true
2,B6,JetBlue Airways,2024-12-10,null,true
1,DL,Delta Air Lines Inc.,2024-12-10,null,true
2,DL,Delta Air Lines Inc.,2024-12-10,null,true
1,EV,Atlantic Southeast Airlines,2024-12-10,null,true
2,EV,Atlantic Southeast Airlines,2024-12-10,null,true


In [0]:
%sql
UPDATE AIRLINES_STAGING
SET LOADED_FLAG = true
WHERE LOADED_FLAG = false;

num_affected_rows
0


In [0]:
%sql
SELECT * FROM AIRLINES_STAGING LIMIT 10;

IATA_CODE,AIRLINE,LOADED_FLAG
UA,United Air Lines Inc.,true
AA,American Airlines Inc.,true
US,US Airways Inc.,true
F9,Frontier Airlines Inc.,true
B6,JetBlue Airways,true
OO,Skywest Airlines Inc.,true
AS,Alaska Airlines Inc.,true
NK,Spirit Air Lines,true
WN,Southwest Airlines Co.,true
DL,Delta Air Lines Inc.,true


###AIRPORTS

In [0]:
%sql
DROP TABLE AIRPORTS_CURATED;
CREATE TABLE AIRPORTS_CURATED (
  AIRPORT_ID INT NOT NULL,
  IATA_CODE STRING NOT NULL,
  AIRPORT STRING NOT NULL,
  CITY STRING,
  STATE STRING,
  COUNTRY STRING,
  LATITUDE FLOAT,
  LONGITUDE FLOAT,
  START_DATE DATE NOT NULL,
  END_DATE DATE,
  IS_ACTIVE BOOLEAN NOT NULL
);


In [0]:
%sql
-- Krok 1: Oznaczam stare rekordy jako nieaktywne
MERGE INTO AIRPORTS_CURATED AS c 
USING (
    SELECT 
        ROW_NUMBER() OVER (PARTITION BY IATA_CODE ORDER BY IATA_CODE) AS AIRPORT_ID,
        IATA_CODE,
        AIRPORT,
        CITY,
        STATE,
        COUNTRY,
        LATITUDE,
        LONGITUDE,
        LOADED_FLAG
    FROM AIRPORTS_STAGING
    WHERE LOADED_FLAG = FALSE
) AS s 
ON c.IATA_CODE = s.IATA_CODE AND c.IS_ACTIVE = TRUE 
WHEN MATCHED AND (
    c.AIRPORT <> s.AIRPORT 
    OR c.CITY <> s.CITY 
    OR c.STATE <> s.STATE 
    OR c.COUNTRY <> s.COUNTRY 
    OR c.LATITUDE <> s.LATITUDE 
    OR c.LONGITUDE <> s.LONGITUDE
) THEN 
    UPDATE SET 
        END_DATE = '2024-12-10',
        IS_ACTIVE = FALSE;

-- Krok 2: Wstawiam nowe rekordy
INSERT INTO AIRPORTS_CURATED (
    AIRPORT_ID, IATA_CODE, AIRPORT, CITY, STATE, COUNTRY, 
    LATITUDE, LONGITUDE, START_DATE, END_DATE, IS_ACTIVE
)
WITH new_records AS (
    SELECT 
        ROW_NUMBER() OVER (PARTITION BY IATA_CODE ORDER BY IATA_CODE) + 
            (SELECT COALESCE(MAX(AIRPORT_ID), 0) FROM AIRPORTS_CURATED) AS AIRPORT_ID,
        IATA_CODE,
        AIRPORT,
        CITY,
        STATE,
        COUNTRY,
        LATITUDE,
        LONGITUDE
    FROM AIRPORTS_STAGING s
    WHERE LOADED_FLAG = FALSE
)
SELECT 
    n.AIRPORT_ID,
    n.IATA_CODE,
    n.AIRPORT,
    n.CITY,
    n.STATE,
    n.COUNTRY,
    n.LATITUDE,
    n.LONGITUDE,
    '2024-12-10' as START_DATE,
    NULL as END_DATE,
    TRUE as IS_ACTIVE
FROM new_records n
LEFT JOIN AIRPORTS_CURATED c 
    ON n.IATA_CODE = c.IATA_CODE AND c.IS_ACTIVE = TRUE
WHERE 
    c.IATA_CODE IS NULL  -- dla nowych rekordów
    OR 
    c.AIRPORT <> n.AIRPORT 
    OR c.CITY <> n.CITY 
    OR c.STATE <> n.STATE 
    OR c.COUNTRY <> n.COUNTRY 
    OR c.LATITUDE <> n.LATITUDE 
    OR c.LONGITUDE <> n.LONGITUDE;  -- dla zmienionych rekordów

-- Aktualizacja flag w staging
UPDATE AIRPORTS_STAGING
SET LOADED_FLAG = TRUE
WHERE LOADED_FLAG = FALSE;

num_affected_rows
322


In [0]:
%sql
SELECT * FROM AIRPORTS_CURATED LIMIT 10;

AIRPORT_ID,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE,START_DATE,END_DATE,IS_ACTIVE
1,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.4404,2024-12-10,null,true
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.6819,2024-12-10,null,true
1,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919,2024-12-10,null,true
1,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183,2024-12-10,null,true
1,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447,2024-12-10,null,true
1,ACK,Nantucket Memorial Airport,Nantucket,MA,USA,41.25305,-70.06018,2024-12-10,null,true
1,ACT,Waco Regional Airport,Waco,TX,USA,31.61129,-97.23052,2024-12-10,null,true
1,ACV,Arcata Airport,Arcata/Eureka,CA,USA,40.97812,-124.10862,2024-12-10,null,true
1,ACY,Atlantic City International Airport,Atlantic City,NJ,USA,39.45758,-74.57717,2024-12-10,null,true
1,ADK,Adak Airport,Adak,AK,USA,51.87796,-176.64603,2024-12-10,null,true


In [0]:
%sql
UPDATE AIRPORTS_STAGING
SET LOADED_FLAG = true
WHERE LOADED_FLAG = false;

num_affected_rows
0


###Distances

In [0]:
%sql
DROP TABLE DISTANCES_CURATED;
CREATE TABLE DISTANCES_CURATED (
  DISTANCE_ID INT NOT NULL,
  ORIGIN_AIRPORT STRING NOT NULL,
  DESTINATION_AIRPORT STRING NOT NULL,
  DISTANCE INT NOT NULL,
  START_DATE DATE NOT NULL,
  END_DATE DATE,
  IS_ACTIVE BOOLEAN NOT NULL
);


In [0]:
%sql
MERGE INTO DISTANCES_CURATED AS c 
USING (
    SELECT 
        ROW_NUMBER() OVER (ORDER BY ORIGIN_AIRPORT, DESTINATION_AIRPORT) as DISTANCE_ID,
        ORIGIN_AIRPORT,
        DESTINATION_AIRPORT,
        DISTANCE,
        LOADED_FLAG
    FROM (
        SELECT DISTINCT 
            ORIGIN_AIRPORT,
            DESTINATION_AIRPORT,
            DISTANCE,
            LOADED_FLAG
        FROM FLIGHTS_STAGING
    ) 
) AS s 
ON c.ORIGIN_AIRPORT = s.ORIGIN_AIRPORT 
   AND c.DESTINATION_AIRPORT = s.DESTINATION_AIRPORT 
   AND c.IS_ACTIVE = TRUE 
WHEN MATCHED 
    AND s.LOADED_FLAG = FALSE 
    AND (c.DISTANCE <> s.DISTANCE) 
THEN 
    UPDATE SET 
        c.END_DATE = '2024-12-10', 
        c.IS_ACTIVE = FALSE 
WHEN NOT MATCHED 
    AND s.LOADED_FLAG = FALSE 
THEN 
    INSERT (
        DISTANCE_ID, 
        ORIGIN_AIRPORT, 
        DESTINATION_AIRPORT, 
        DISTANCE, 
        START_DATE, 
        END_DATE, 
        IS_ACTIVE
    ) 
    VALUES (
        s.DISTANCE_ID, 
        s.ORIGIN_AIRPORT, 
        s.DESTINATION_AIRPORT, 
        s.DISTANCE, 
        '2024-12-10', 
        NULL, 
        TRUE
    )

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4298,0,0,4298


In [0]:
%sql
SELECT * FROM DISTANCES_CURATED LIMIT 10;

DISTANCE_ID,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DISTANCE,START_DATE,END_DATE,IS_ACTIVE
1,ABE,ATL,692,2024-12-10,null,true
2,ABE,DTW,425,2024-12-10,null,true
3,ABE,ORD,654,2024-12-10,null,true
4,ABI,DFW,158,2024-12-10,null,true
5,ABQ,ATL,1269,2024-12-10,null,true
6,ABQ,BWI,1670,2024-12-10,null,true
7,ABQ,DAL,580,2024-12-10,null,true
8,ABQ,DEN,349,2024-12-10,null,true
9,ABQ,DFW,569,2024-12-10,null,true
10,ABQ,HOU,759,2024-12-10,null,true


###Date

In [0]:
%sql
--sprawdzenie skrajnych dat
SELECT 
    MIN(CAST(CONCAT(YEAR, '-', MONTH, '-', DAY) AS DATE)) AS MIN_DATE,
    MAX(CAST(CONCAT(YEAR, '-', MONTH, '-', DAY) AS DATE)) AS MAX_DATE
FROM FLIGHTS_STAGING;


MIN_DATE,MAX_DATE
2015-01-01,2015-03-10


In [0]:
%sql
DROP TABLE DIM_DATE;
CREATE TABLE DIM_DATE (
    DATE_ID INT NOT NULL,
    DATE DATE NOT NULL,
    YEAR INT NOT NULL,
    MONTH INT NOT NULL, 
    DAY INT NOT NULL,  
    DAY_OF_WEEK INT NOT NULL
);


In [0]:
%sql
INSERT INTO DIM_DATE
WITH date_sequence AS (
  SELECT explode(sequence(
    (SELECT MIN(CAST(CONCAT(YEAR, '-', MONTH, '-', DAY) AS DATE)) FROM FLIGHTS_STAGING),
    (SELECT MAX(CAST(CONCAT(YEAR, '-', MONTH, '-', DAY) AS DATE)) FROM FLIGHTS_STAGING),
    interval 1 day
  )) as date
)
SELECT 
  CAST(DATE_FORMAT(date, 'yyyyMMdd') AS INT) as DATE_ID,
  date as DATE,
  YEAR(date) as YEAR,
  MONTH(date) as MONTH,
  DAY(date) as DAY,
  DAYOFWEEK(date) as DAY_OF_WEEK
FROM date_sequence;

num_affected_rows,num_inserted_rows
69,69


In [0]:
%sql
SELECT * FROM DIM_DATE LIMIT 10;

DATE_ID,DATE,YEAR,MONTH,DAY,DAY_OF_WEEK
20150101,2015-01-01,2015,1,1,5
20150102,2015-01-02,2015,1,2,6
20150103,2015-01-03,2015,1,3,7
20150104,2015-01-04,2015,1,4,1
20150105,2015-01-05,2015,1,5,2
20150106,2015-01-06,2015,1,6,3
20150107,2015-01-07,2015,1,7,4
20150108,2015-01-08,2015,1,8,5
20150109,2015-01-09,2015,1,9,6
20150110,2015-01-10,2015,1,10,7


In [0]:
%sql
DROP TABLE FACTS_FLIGHTS;
CREATE TABLE FACTS_FLIGHTS (
    FLIGHT_ID INT NOT NULL,
    DATE_ID INT NOT NULL,
    AIRLINE_CODE STRING NOT NULL,
    DISTANCE_ID INT NOT NULL,  
    FLIGHT_NUMBER STRING NOT NULL,
    TAIL_NUMBER STRING,
    SCHEDULED_DEPARTURE STRING NOT NULL, 
    DEPARTURE_TIME STRING,     
    DEPARTURE_DELAY INT,
    TAXI_OUT INT,
    WHEELS_OFF STRING,      
    SCHEDULED_TIME INT,
    ELAPSED_TIME INT,
    AIR_TIME INT,
    WHEELS_ON STRING,        
    TAXI_IN INT,
    SCHEDULED_ARRIVAL STRING,  
    ARRIVAL_TIME STRING,       
    ARRIVAL_DELAY INT,
    DIVERTED BOOLEAN NOT NULL,
    CANCELLED BOOLEAN,
    CANCELLATION_REASON STRING,
    AIR_SYSTEM_DELAY INT,
    SECURITY_DELAY INT,
    AIRLINE_DELAY INT,
    LATE_AIRCRAFT_DELAY INT,
    WEATHER_DELAY INT
);


In [0]:
%sql
DROP TABLE FACTS_FLIGHTS_HISTORY;
CREATE TABLE FACTS_FLIGHTS_HISTORY (
    FLIGHT_ID INT NOT NULL,
    DATE_ID INT NOT NULL,
    AIRLINE_CODE STRING NOT NULL,
    DISTANCE_ID INT NOT NULL,  
    FLIGHT_NUMBER STRING NOT NULL,
    TAIL_NUMBER STRING,
    SCHEDULED_DEPARTURE STRING NOT NULL, 
    DEPARTURE_TIME STRING,     
    DEPARTURE_DELAY INT,
    TAXI_OUT INT,
    WHEELS_OFF STRING,      
    SCHEDULED_TIME INT,
    ELAPSED_TIME INT,
    AIR_TIME INT,
    WHEELS_ON STRING,        
    TAXI_IN INT,
    SCHEDULED_ARRIVAL STRING,  
    ARRIVAL_TIME STRING,       
    ARRIVAL_DELAY INT,
    DIVERTED BOOLEAN NOT NULL,
    CANCELLED BOOLEAN,
    CANCELLATION_REASON STRING,
    AIR_SYSTEM_DELAY INT,
    SECURITY_DELAY INT,
    AIRLINE_DELAY INT,
    LATE_AIRCRAFT_DELAY INT,
    WEATHER_DELAY INT
);


In [0]:
%sql
INSERT INTO FACTS_FLIGHTS 
SELECT    
    ROW_NUMBER() OVER (ORDER BY s.YEAR, s.MONTH, s.DAY, s.FLIGHT_NUMBER, s.SCHEDULED_DEPARTURE) as FLIGHT_ID,
    CAST(DATE_FORMAT(CAST(CONCAT(s.YEAR, '-', s.MONTH, '-', s.DAY) AS DATE), 'yyyyMMdd') AS INT) as DATE_ID,
    s.AIRLINE as AIRLINE_CODE,
    d.DISTANCE_ID,
    s.FLIGHT_NUMBER,
    s.TAIL_NUMBER,
    s.SCHEDULED_DEPARTURE,
    s.DEPARTURE_TIME,
    s.DEPARTURE_DELAY,
    s.TAXI_OUT,
    s.WHEELS_OFF,
    s.SCHEDULED_TIME,
    s.ELAPSED_TIME,
    s.AIR_TIME,
    s.WHEELS_ON,
    s.TAXI_IN,
    s.SCHEDULED_ARRIVAL,
    s.ARRIVAL_TIME,
    s.ARRIVAL_DELAY,
    s.DIVERTED,
    s.CANCELLED,
    s.CANCELLATION_REASON,
    s.AIR_SYSTEM_DELAY,
    s.SECURITY_DELAY,
    s.AIRLINE_DELAY,
    s.LATE_AIRCRAFT_DELAY,
    s.WEATHER_DELAY 
FROM FLIGHTS_STAGING s 
LEFT JOIN DISTANCES_CURATED d     
    ON s.ORIGIN_AIRPORT = d.ORIGIN_AIRPORT     
    AND s.DESTINATION_AIRPORT = d.DESTINATION_AIRPORT     
    AND d.IS_ACTIVE = TRUE 
WHERE s.LOADED_FLAG = FALSE;

num_affected_rows,num_inserted_rows
1048575,1048575


In [0]:
%sql
SELECT * FROM FACTS_FLIGHTS LIMIT 10;

FLIGHT_ID,DATE_ID,AIRLINE_CODE,DISTANCE_ID,FLIGHT_NUMBER,TAIL_NUMBER,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
1,20150101,VX,3822,1,N837VA,08:00,07:52,-8,11,08:03,300,286,271,15:34,4,16:00,15:38,-22,false,false,null,null,null,null,null,null
2,20150101,AS,942,1,N523AS,08:00,07:56,-4,8,08:04,360,341,328,10:32,5,11:00,10:37,-23,false,false,null,null,null,null,null,null
3,20150101,HA,2249,1,N396HA,08:40,08:40,0,12,08:52,345,364,347,12:39,5,12:25,12:44,19,false,false,null,0,0,19,0,0
4,20150101,AA,2084,1,N787AA,09:00,08:55,-5,17,09:12,390,402,378,12:30,7,12:30,12:37,7,false,false,null,null,null,null,null,null
5,20150101,B6,2076,1,N715JB,09:50,10:13,23,22,10:35,186,181,153,13:08,6,12:56,13:14,18,false,false,null,0,0,18,0,0
6,20150101,HA,1698,10,N389HA,08:00,08:10,10,21,08:31,325,303,272,15:03,10,15:25,15:13,-12,false,false,null,null,null,null,null,null
7,20150101,WN,862,10,N206WN,10:00,10:14,14,8,10:22,165,144,132,13:34,4,13:45,13:38,-7,false,false,null,null,null,null,null,null
8,20150101,AA,2257,10,N796AA,21:50,21:50,0,14,22:04,309,294,275,05:39,5,05:59,05:44,-15,false,false,null,null,null,null,null,null
9,20150101,B6,2246,100,N594JB,22:37,22:37,0,9,22:46,282,289,274,06:20,6,06:19,06:26,7,false,false,null,null,null,null,null,null
10,20150101,UA,3093,1001,N23721,07:38,07:35,-3,12,07:47,125,105,87,10:14,6,10:43,10:20,-23,false,false,null,null,null,null,null,null


##ANALIZY

###Zarejestrowane trasy

In [0]:
%sql
SELECT
  d.DATE, 
  COUNT(f.DATE_ID) AS NUMBER_OF_FLIGHTS
FROM 
  FACTS_FLIGHTS f
JOIN 
  Dim_Date d ON f.DATE_ID = d.DATE_ID
GROUP BY 
  d.DATE
ORDER BY 
  d.DATE; 


DATE,NUMBER_OF_FLIGHTS
2015-01-01,13950
2015-01-02,16741
2015-01-03,15434
2015-01-04,16352
2015-01-05,16548
2015-01-06,15315
2015-01-07,15571
2015-01-08,16009
2015-01-09,16008
2015-01-10,12344


In [0]:
df = spark.sql("""
    SELECT
    d.DATE, 
    COUNT(f.DATE_ID) AS NUMBER_OF_FLIGHTS
    FROM 
    FACTS_FLIGHTS f
    JOIN 
    Dim_Date d ON f.DATE_ID = d.DATE_ID
    GROUP BY 
    d.DATE
    ORDER BY 
    d.DATE; 
""")


In [0]:
stats1 = df.toPandas()

In [0]:
import plotly.express as px

# Tworzymy wykres słupkowy
fig = px.line(stats1, 
             x='DATE', 
             y='NUMBER_OF_FLIGHTS', 
             title='Liczba lotów dla każdego DATE_ID', 
             labels={'DATE': 'Data', 'NUMBER_OF_FLIGHTS': 'Liczba lotów'},
             color_discrete_sequence=['purple'])

fig.update_layout(
    width=800,
    height=600
)

fig.show()


###Opóźnienia

In [0]:
%sql
SELECT
  AVG(DEPARTURE_DELAY)
FROM 
  FACTS_FLIGHTS

avg(DEPARTURE_DELAY)
11.334851247695875


In [0]:
%sql
SELECT 
    a.AIRPORT,  
    AVG(f.DEPARTURE_DELAY) AS AVERAGE_DELAY  
FROM 
    FACTS_FLIGHTS f
JOIN 
    DISTANCES_CURATED d ON f.DISTANCE_ID = d.DISTANCE_ID 
JOIN 
    AIRPORTS_CURATED a ON d.ORIGIN_AIRPORT = a.IATA_CODE 
GROUP BY 
    a.AIRPORT  
ORDER BY 
    AVERAGE_DELAY DESC;  


AIRPORT,AVERAGE_DELAY
Trenton Mercer Airport,39.76171079429735
Wilmington Airport,34.71153846153846
Southwest Oregon Regional Airport (North Bend Municipal),26.305555555555557
University Park Airport,24.116788321167885
Aberdeen Regional Airport,23.261194029850746
Santa Maria Public Airport (Capt G. Allan Hancock Field),22.984962406015036
Jack Brooks Regional Airport (Southeast Texas Regional),22.84269662921348
Del Norte County Airport (Jack McNamara Field),22.311475409836067
Bangor International Airport,21.9
University of Illinois - Willard Airport,21.63054187192118


In [0]:
df2 = spark.sql("""
    SELECT 
        a.AIRPORT,  
        AVG(f.DEPARTURE_DELAY) AS AVERAGE_DELAY  
    FROM 
        FACTS_FLIGHTS f
    JOIN 
        DISTANCES_CURATED d ON f.DISTANCE_ID = d.DISTANCE_ID 
    JOIN 
        AIRPORTS_CURATED a ON d.ORIGIN_AIRPORT = a.IATA_CODE 
    GROUP BY 
        a.AIRPORT  
    ORDER BY 
        AVERAGE_DELAY DESC;  

""")


In [0]:
stats2 = df2.toPandas().head(10)

In [0]:
fig = px.bar(stats2, 
             x='AIRPORT', 
             y='AVERAGE_DELAY', 
             title='Top 10 najbardziej opóźnionych lotnisk :oo', 
             labels={'AIRPORT': 'AIRPORT', 'AVERAGE_DELAY': 'AVERAGE_DELAY'},
             color_discrete_sequence=['pink'])

fig.update_layout(
    width=800,
    height=600
)

fig.show()


In [0]:
stats3 = df2.toPandas().tail(10)

In [0]:
fig = px.bar(stats3, 
             x='AIRPORT', 
             y='AVERAGE_DELAY', 
             title='Top 10 najmniej opóźnionych lotnisk :oo', 
             labels={'AIRPORT': 'AIRPORT', 'AVERAGE_DELAY': 'AVERAGE_DELAY'},
             color_discrete_sequence=['pink'])

fig.update_layout(
    width=800,
    height=600
)

fig.show()


###Przykład ładowania przyrostowego i obsługi SCD2 dla danych airports:

In [0]:
airports_df2 = spark.read.format("csv") \
    .option("header", "true") \
    .schema(airports_schema) \
    .load("/FileStore/tables/airports-2.csv")


In [0]:
from pyspark.sql.functions import col, when
airports_df2 = airports_df2.withColumn("LOADED_FLAG", when(col("LOADED_FLAG") == 1, True).otherwise(False))

In [0]:
airports_df2.write.mode("overwrite").saveAsTable("AIRPORTS_STAGING")

In [0]:
%sql
SELECT * FROM AIRPORTS_STAGING LIMIT 10;

IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE,LOADED_FLAG
ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.4404,true
ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.6819,true
ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919,true
ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183,true
ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447,true
ACK,Nantucket Memorial Airport,Nantucket,MA,USA,41.25305,-70.06018,true
ACT,Waco Regional Airport,Waco,TX,USA,31.61129,-97.23052,true
ACV,Arcata Airport,Arcata/Eureka,CA,USA,40.97812,-124.10862,true
ACY,Atlantic City International Airport,Atlantic City,NJ,USA,39.45758,-74.57717,true
ADK,Adak Airport,Adak,AK,USA,51.87796,-176.64603,true


In [0]:
%sql
-- Krok 1: Oznaczam stare rekordy jako nieaktywne
MERGE INTO AIRPORTS_CURATED AS c 
USING (
    SELECT 
        ROW_NUMBER() OVER (PARTITION BY IATA_CODE ORDER BY IATA_CODE) AS AIRPORT_ID,
        IATA_CODE,
        AIRPORT,
        CITY,
        STATE,
        COUNTRY,
        LATITUDE,
        LONGITUDE,
        LOADED_FLAG
    FROM AIRPORTS_STAGING
    WHERE LOADED_FLAG = FALSE
) AS s 
ON c.IATA_CODE = s.IATA_CODE AND c.IS_ACTIVE = TRUE 
WHEN MATCHED AND (
    c.AIRPORT <> s.AIRPORT 
    OR c.CITY <> s.CITY 
    OR c.STATE <> s.STATE 
    OR c.COUNTRY <> s.COUNTRY 
    OR c.LATITUDE <> s.LATITUDE 
    OR c.LONGITUDE <> s.LONGITUDE
) THEN 
    UPDATE SET 
        END_DATE = '2024-12-10',
        IS_ACTIVE = FALSE;

-- Krok 2: Wstawiam nowe rekordy
INSERT INTO AIRPORTS_CURATED (
    AIRPORT_ID, IATA_CODE, AIRPORT, CITY, STATE, COUNTRY, 
    LATITUDE, LONGITUDE, START_DATE, END_DATE, IS_ACTIVE
)
WITH new_records AS (
    SELECT 
        ROW_NUMBER() OVER (PARTITION BY IATA_CODE ORDER BY IATA_CODE) + 
            (SELECT COALESCE(MAX(AIRPORT_ID), 0) FROM AIRPORTS_CURATED) AS AIRPORT_ID,
        IATA_CODE,
        AIRPORT,
        CITY,
        STATE,
        COUNTRY,
        LATITUDE,
        LONGITUDE
    FROM AIRPORTS_STAGING s
    WHERE LOADED_FLAG = FALSE
)
SELECT 
    n.AIRPORT_ID,
    n.IATA_CODE,
    n.AIRPORT,
    n.CITY,
    n.STATE,
    n.COUNTRY,
    n.LATITUDE,
    n.LONGITUDE,
    '2024-12-10' as START_DATE,
    NULL as END_DATE,
    TRUE as IS_ACTIVE
FROM new_records n
LEFT JOIN AIRPORTS_CURATED c 
    ON n.IATA_CODE = c.IATA_CODE AND c.IS_ACTIVE = TRUE
WHERE 
    c.IATA_CODE IS NULL  -- dla nowych rekordów
    OR 
    c.AIRPORT <> n.AIRPORT 
    OR c.CITY <> n.CITY 
    OR c.STATE <> n.STATE 
    OR c.COUNTRY <> n.COUNTRY 
    OR c.LATITUDE <> n.LATITUDE 
    OR c.LONGITUDE <> n.LONGITUDE;  -- dla zmienionych rekordów

-- Aktualizacja flag w staging
UPDATE AIRPORTS_STAGING
SET LOADED_FLAG = TRUE
WHERE LOADED_FLAG = FALSE;

num_affected_rows
4


In [0]:
%sql
SELECT * 
FROM AIRPORTS_CURATED
WHERE CITY == "Vernal"



AIRPORT_ID,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE,START_DATE,END_DATE,IS_ACTIVE
2,VEL,ValdezNEWVAL Airport,Vernal,UTNV,USA,40.4409,-109.50992,2024-12-10,null,true
1,VEL,Valdez Airport,Vernal,UT,USA,40.4409,-109.50992,2024-12-10,2024-12-10,false


In [0]:
%sql
SELECT * 
FROM AIRPORTS_CURATED
WHERE CITY == "NewNew"

AIRPORT_ID,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE,START_DATE,END_DATE,IS_ACTIVE
2,NEW,New Airprot,NewNew,NW,PL,39.65658,-144.60597,2024-12-10,null,true


In [0]:
%sql
SELECT * 
FROM AIRPORTS_CURATED
WHERE IATA_CODE == "YUM"

AIRPORT_ID,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE,START_DATE,END_DATE,IS_ACTIVE
1,YUM,Yuma International Airport,Yuma,AZ,USA,32.65658,-114.60597,2024-12-10,null,true
